# Fresh AI: train `fresh-unified-1` on Kaggle (run 3)

Same training as the Colab notebook, but self-contained: it rebuilds the data from the repo, so it does not need Google Drive.

## Setup (once, in the notebook settings on the right/top menu)
1. **Accelerator:** choose **GPU** (T4 x2 or P100). Training uses one GPU.
2. **Internet:** turn **On** (needs a verified Kaggle phone number).
3. Run the notebook one of two ways:
   - **Best:** *Save Version → Save & Run All (Commit)*. It runs in the background for hours; you can close the app. The result stays under the version's **Output** tab (checkpoint, manifest, samples).
   - Or run cells top to bottom (the session must stay open; files are lost if it ends).

It stops with an error at the first problem (including a check that the first loss is ~10, which proves the initialization fix). Takes about 3 to 5 hours.

**Honest expectation:** ~50M parameters on ~33M tokens is a small model. It will not be reliable or superintelligent; this is stage 1. The checkpoint is never marked production-ready.

In [ ]:
# 1. Setup
import subprocess, re, os, sys
W = "/kaggle/working"
def run(cmd, check=True):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode, "".join(lines)

rc, out = run("nvidia-smi -L", check=False)
if rc: raise RuntimeError("No GPU: set Accelerator to GPU in the notebook settings.")
if not os.path.exists(f"{W}/fresh-web-lite"):
    run(f"git clone --depth 1 https://github.com/kanarafat54-boop/fresh-web-lite.git {W}/fresh-web-lite")
else:
    run(f"git -C {W}/fresh-web-lite pull --ff-only")
os.chdir(f"{W}/fresh-web-lite")
run("pip -q install tokenizers numpy")
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 2. Build data from the repo: mix, tokenizer, token streams (about 10 to 15 minutes)
D = f"{W}/data"
os.makedirs(D, exist_ok=True)
if not os.path.exists(f"{D}/mix/train.jsonl.gz"):
    run(f"python3 training/build_training_mix.py --recipe training/mix_recipe_v1.json --out-dir {D}/mix")
if not os.path.exists(f"{D}/tokenizer-v2.json"):
    run(f"python3 training/fresh_tokenizer_v2.py train --mix-train {D}/mix/train.jsonl.gz --output {D}/tokenizer-v2.json")
for split in ("train", "validation"):
    if not os.path.exists(f"{D}/{split}.bin"):
        run(f"python3 training/fresh_tokenizer_v2.py encode --tokenizer {D}/tokenizer-v2.json --input {D}/mix/{split}.jsonl.gz --output {D}/{split}.bin")
run(f"ls -lh {D}")

In [ ]:
# 3. Smoke test (tiny model, 20 steps). Tries --compile first, falls back automatically. Checks the init fix.
base = (f"python3 training/train_fresh_unified_full.py --train-bin {D}/train.bin --val-bin {D}/validation.bin --tokenizer {D}/tokenizer-v2.json "
        f"--spec training/model_spec_smoke.json --seq-len 256 --micro-batch 4 --accum 1 --max-steps 20 --warmup 5 --eval-every 10 --save-every 20 "
        f"--eval-blocks 8 --out-dir {W}/smoke --run-id smoke")
COMPILE = " --compile"
run(f"rm -rf {W}/smoke")
rc, out = run(base + COMPILE, check=False)
if rc:
    print("\n--compile failed; retrying without it")
    COMPILE = ""
    run(f"rm -rf {W}/smoke")
    rc, out = run(base, check=False)
if rc: raise RuntimeError("Smoke test failed even without --compile. Send the error to Claude.")
m = re.search(r"step 1/\d+ loss ([0-9.]+)", out)
if not m or float(m.group(1)) > 20:
    raise RuntimeError(f"First loss {m.group(1) if m else 'missing'} is too high: initialization fix not active.")
print(f"\nSMOKE OK. First loss {m.group(1)} (expected ~10-11). compile flag: '{COMPILE.strip() or 'off'}'")

In [ ]:
# 4. REAL TRAINING (about 3 to 5 hours). Saves a resumable checkpoint every 50 steps.
cmd = (f"python3 training/train_fresh_unified_full.py --train-bin {D}/train.bin --val-bin {D}/validation.bin --tokenizer {D}/tokenizer-v2.json "
       f"--spec training/model_spec.json --seq-len 1024 --micro-batch 8 --accum 8 --epochs 3 --save-every 50 --eval-every 200{COMPILE} "
       f"--mix-manifest {D}/mix/manifest.json --out-dir {W}/train-run3 --run-id fresh-unified-1-run3")
run(cmd)

In [ ]:
# 5. Results (the manifest and samples are the evidence). Send this output to Claude.
run(f"cat {W}/train-run3/fresh-unified-1-run3.manifest.json; cat {W}/train-run3/fresh-unified-1-run3.samples.txt")
run(f"rm -f {W}/train-run3/last.pt; cp {D}/tokenizer-v2.json {D}/tokenizer-v2.manifest.json {D}/mix/manifest.json {W}/train-run3/ 2>/dev/null; ls -lh {W}/train-run3")